# Synthetic Data Generation for Cereal CO2 Optimization

## Overview
This notebook demonstrates the complete pipeline for generating synthetic training data for the NEAT-based CO2 optimization model. The process consists of:

1. **Causal seed generation**: Create realistic base records with thermodynamically-accurate process temperatures per strategy
2. **Environmental impact engine**: Calculate CO2 emissions based on causal parameters
3. **Generative model training**: Train CTGAN to learn and extend the data distribution
4. **Synthetic data synthesis**: Scale up to 50,000 realistic scenarios
5. **Data cleaning and export**: Ensure data quality and prepare for training

All parameters match the production pipeline schema, and reflect real-world thermal requirements (Animal feed: 60°C, Composting: 60°C, Biochar: 450°C, Biomass combustion: 900°C).

In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
from ctgan import CTGAN

# Set deterministic behavior for reproducibility
np.random.seed(42)

# 1. Causal Seed Generation

Create a realistic base dataset of 1,500 observations derived from agronomic principles (Castilla y León 2018 agricultural reference data).

Each record represents a real-world scenario linking:
- **Subproduct type** (Husk, Bran, Straw, Silo dust) → determines typical moisture & volume
- **Season** (Dry, Rainy) → conditions seasonal humidity variations
- **Generated volume** → calibrated from field measurements
- **Moisture percentage** → critical factor for post-harvest processing
- **Process temperature** → reflects thermodynamic requirements of each strategy (NOT generic)
- **Reuse strategy** → assigned first, then temperature determined by strategy physics

This seed serves as the training foundation for CTGAN to learn realistic multivariate distributions.

**Temperature calibration by strategy (realistic thermodynamic values):**
- **Animal feed**: 50-70°C (pasteurization: eliminate pathogens while preserving nutrients)
- **Composting**: 52-68°C (thermophilic aerobic respiration: natural decomposition)
- **Biochar**: 350-550°C (pyrolysis: thermal carbonization for carbon sequestration)
- **Biomass combustion**: 750-1050°C (complete oxidation: energy release and power generation)

In [2]:
n_real_samples = 1500  # Increased to capture multivariate causality better

# Load data-generation parameters from JSON config
config_candidates = [
    Path("../config/data_generation_params.json"),
    Path("config/data_generation_params.json")
 ]
config_path = next((p for p in config_candidates if p.exists()), None)
if config_path is None:
    raise FileNotFoundError("Could not find config/data_generation_params.json")

with open(config_path, "r", encoding="utf-8") as f:
    data_params = json.load(f)

# Read categorical and physical generation parameters
seasons_catalog = data_params["seasons"]
subproducts_catalog = data_params["subproducts"]
strategies = data_params["strategies"]
physical_minimum = data_params["physical_minimum"]
volume_by_subproduct = data_params["volume_by_subproduct"]
humidity_by_subproduct_and_season = data_params["humidity_by_subproduct_and_season"]
strategy_temperatures = data_params["temperature_by_strategy"]

# Sample categorical variables
seasons = np.random.choice(seasons_catalog, n_real_samples)
subproduct_types = np.random.choice(subproducts_catalog, n_real_samples)

# Generate causal data
causal_data = []

for subproduct, season in zip(subproduct_types, seasons):
    humidity_cfg = humidity_by_subproduct_and_season[subproduct][season]
    volume_cfg = volume_by_subproduct[subproduct]

    moisture = np.random.normal(humidity_cfg["mean"], humidity_cfg["std"])
    volume = np.random.normal(volume_cfg["mean"], volume_cfg["std"] )

    # Select strategy first, then assign temperature based on thermodynamic requirements
    strategy = np.random.choice(strategies)
    temp_params = strategy_temperatures[strategy]
    temperature = np.random.normal(temp_params['mean'], temp_params['std'])

    causal_data.append({
        'subproduct_type': subproduct,
        'season': season,
        'generated_volume_tons': max(physical_minimum, volume),
        'moisture_pct': max(physical_minimum, moisture),
        'process_temperature_c': temperature,
        'reuse_strategy': strategy
    })

df_seed = pd.DataFrame(causal_data)

print(f"Causal seed dataset generated. Shape: {df_seed.shape}")
print(f"Columns: {list(df_seed.columns)}")
print(f"\nLoaded config from: {config_path}")
print(f"\nFirst 3 rows:\n{df_seed.head(3)}")
print(f"\nTemperature ranges by strategy:")
for strat in strategies:
    temps = df_seed[df_seed['reuse_strategy'] == strat]['process_temperature_c']
    print(f"  {strat:24} -> mean={temps.mean():7.1f}C, std={temps.std():6.1f}C")

Causal seed dataset generated. Shape: (1500, 6)
Columns: ['subproduct_type', 'season', 'generated_volume_tons', 'moisture_pct', 'process_temperature_c', 'reuse_strategy']

Loaded config from: ..\config\data_generation_params.json

First 3 rows:
  subproduct_type season  generated_volume_tons  moisture_pct  \
0           Straw    Dry              55.463770      5.001189   
1            Bran  Rainy              14.250061     17.274424   
2            Bran    Dry              17.877343     17.039919   

   process_temperature_c      reuse_strategy  
0             913.441224  Biomass combustion  
1              46.925637          Composting  
2              61.794168         Animal feed  

Temperature ranges by strategy:
  Biomass combustion       -> mean=  887.5C, std= 142.6C
  Animal feed              -> mean=   60.3C, std=  10.4C
  Composting               -> mean=   60.2C, std=   7.6C
  Biochar                  -> mean=  461.1C, std= 101.2C


## 2. Environmental Impact Engine

Calculate CO2 emissions based on the causal parameters. This models the physical and biological processes that determine environmental impact for each reuse strategy.

**Emission logic:**
- **Base emission**: Thermal processing intensity × volume
- **Biomass combustion**: Adds moisture penalty (expensive if wet)
- **Animal feed**: Higher penalties for unsuitable subproducts or high moisture
- **Biochar**: Low emissions if dry (<10%), penalty if wet
- **Composting**: Fixed factor plus volume-based biological respiration

This target serves as the fitness function for neuroevolution.

In [3]:
def calculate_co2_emissions(row):
    """Calculate CO2 emissions based on strategy and physical parameters."""
    # Base emission from thermal processing
    base_emission = (row['process_temperature_c'] * 0.5) * row['generated_volume_tons']
    
    if row['reuse_strategy'] == 'Biomass combustion':
        # Moisture increases combustion complexity and incomplete reactions
        moisture_penalty = (row['moisture_pct'] ** 1.5) * 2
        return base_emission + moisture_penalty - 50 
        
    elif row['reuse_strategy'] == 'Animal feed':
        # Some subproducts unsuitable for feed; high moisture degrades nutritional value
        if row['subproduct_type'] in ['Husk', 'Straw', 'Silo dust'] or row['moisture_pct'] > 18:
            return base_emission * 1.8  # Higher emissions for poor fit
        return base_emission * 0.4  # Low emissions for good fit
        
    elif row['reuse_strategy'] == 'Biochar':
        # Carbonization efficiency inversely related to moisture
        if row['moisture_pct'] < 10:
            return base_emission * 0.2  # Excellent: dry material
        return base_emission * 1.2  # Poor: wet material
        
    elif row['reuse_strategy'] == 'Composting':
        # Biological oxidation creates steady emissions
        return base_emission * 0.8 + (row['generated_volume_tons'] * 5)

# Compute target metrics
df_seed['co2_emissions_kg'] = df_seed.apply(calculate_co2_emissions, axis=1)
df_seed['co2_per_ton'] = df_seed['co2_emissions_kg'] / df_seed['generated_volume_tons']

print(f"Emissions calculated. Dataset shape: {df_seed.shape}")
print(f"\nEmissions statistics (kg CO2):")
print(df_seed['co2_emissions_kg'].describe())

Emissions calculated. Dataset shape: (1500, 8)

Emissions statistics (kg CO2):
count     1500.000000
mean      4114.115671
std       5856.375168
min          1.037424
25%        485.301132
50%       1411.902662
75%       5151.894187
max      29786.104037
Name: co2_emissions_kg, dtype: float64


## 3. Generative Model Training (CTGAN)

Train a Conditional TabularGAN (CTGAN) to learn the multivariate distribution of the causal seed and extend it to synthetic data. This allows us to:
- **Preserve correlations**: CTGAN captures dependencies between categorical and continuous features
- **Extend coverage**: Generate new realistic combinations not in the original seed
- **Scale up**: Produce 50,000 diverse scenarios for NEAT training

**Configuration:**
- **Epochs**: 100 (adjust to 300+ for production)
- **Discrete columns**: `subproduct_type`, `season`, `reuse_strategy` (CTGAN's Gumbel-Softmax learns these)

In [4]:
discrete_columns = ['subproduct_type', 'season', 'reuse_strategy']

print("Training CTGAN generative model...")
print(f"Input shape: {df_seed.shape}")
print(f"Discrete columns: {discrete_columns}\n")

# Initialize and train CTGAN
ctgan_model = CTGAN(epochs=100, verbose=True)
ctgan_model.fit(df_seed, discrete_columns)

print("\nCTGAN training completed.")

Training CTGAN generative model...
Input shape: (1500, 8)
Discrete columns: ['subproduct_type', 'season', 'reuse_strategy']



Gen. (-00.60) | Discrim. (-00.19): 100%|██████████| 100/100 [00:07<00:00, 14.00it/s]


CTGAN training completed.


## 4. Synthetic Data Synthesis and Post-Generation Cleaning

Generate 50,000 synthetic scenarios from the trained CTGAN. Then perform mandatory post-processing to:
- **Fix pathological GAN artifacts**: Clip negative or impossible values (volumes/moisture < 0.1)
- **Restore thermodynamic consistency**: Reassign realistic process temperatures per strategy (CTGAN cannot preserve complex correlations)
- **Ensure physical validity**: Recalculate CO2 emissions so NEAT learns unbroken causal chains
- **Maximize search space**: Create diverse, realistic scenarios for NEAT population initialization

In [5]:
n_synthetic = 50000
print(f"Generating {n_synthetic} synthetic scenarios for genetic algorithm search space...")

df_synthetic = ctgan_model.sample(n_synthetic)

print(f"Synthetic dataset generated. Shape: {df_synthetic.shape}")
print(f"First 2 rows:\n{df_synthetic.head(2)}\n")

# Post-generation cleaning: Fix pathological GAN artifacts
physical_columns = ['generated_volume_tons', 'moisture_pct']

print("Applying post-generation cleaning...")
print(f"  Clipping {physical_columns} to logical lower bound ({physical_minimum})...")
df_synthetic[physical_columns] = df_synthetic[physical_columns].clip(lower=physical_minimum)

# CRITICAL: Ensure thermodynamic consistency - reassign realistic process temperatures per strategy
print("  Reassigning realistic process temperatures per strategy (thermodynamic consistency)...")
for strategy in df_synthetic['reuse_strategy'].unique():
    mask = df_synthetic['reuse_strategy'] == strategy
    temp_params = strategy_temperatures[strategy]
    df_synthetic.loc[mask, 'process_temperature_c'] = np.random.normal(
        temp_params['mean'], temp_params['std'], size=mask.sum()
    )

# Recalculate emissions on cleaned data to ensure physical consistency
print("  Recalculating CO2 emissions on cleaned and thermodynamically-corrected inputs...")
df_synthetic['co2_emissions_kg'] = df_synthetic.apply(calculate_co2_emissions, axis=1)
df_synthetic['co2_per_ton'] = df_synthetic['co2_emissions_kg'] / df_synthetic['generated_volume_tons']

print(f"\nCleaning completed. Final dataset shape: {df_synthetic.shape}")
print(f"\nGenerated volume statistics (tons):")
print(df_synthetic['generated_volume_tons'].describe())

# Verify thermodynamic consistency
print(f"\nProcess temperature ranges (now physically coherent):")
for strategy in sorted(df_synthetic['reuse_strategy'].unique()):
    temps = df_synthetic[df_synthetic['reuse_strategy'] == strategy]['process_temperature_c']
    print(f"  {strategy:24} -> mean={temps.mean():7.1f}C, std={temps.std():6.1f}C")

Generating 50000 synthetic scenarios for genetic algorithm search space...
Synthetic dataset generated. Shape: (50000, 8)
First 2 rows:
  subproduct_type season  generated_volume_tons  moisture_pct  \
0            Husk  Rainy              26.073744     10.500494   
1           Straw    Dry              46.680400     10.994034   

   process_temperature_c reuse_strategy  co2_emissions_kg  co2_per_ton  
0             685.681457        Biochar        687.571299    53.801345  
1             272.449919        Biochar       1148.229309   439.433787  

Applying post-generation cleaning...
  Clipping ['generated_volume_tons', 'moisture_pct'] to logical lower bound (0.1)...
  Reassigning realistic process temperatures per strategy (thermodynamic consistency)...
  Recalculating CO2 emissions on cleaned and thermodynamically-corrected inputs...

Cleaning completed. Final dataset shape: (50000, 8)

Generated volume statistics (tons):
count    50000.000000
mean        22.636733
std         17.86528

## 5. Export for Training

Save the cleaned synthetic dataset to CSV. This file serves as the input to:
- **Preprocessing**: Train/test split and feature scaling
- **NEAT training**: 50+ generation cycles of neuroevolution
- **Baseline inference**: CO2 reduction evaluation

The output file includes all 9 columns in canonical English naming, ready for the production pipeline.

In [6]:
output_file = "../data/processed/dataset_optimization_cereal_co2.csv"

print(f"Exporting synthetic training dataset...")
df_synthetic.to_csv(output_file, index=False)

print(f"✓ Training dataset successfully saved to: {output_file}")
print(f"\nFinal dataset summary:")
print(f"  Rows: {len(df_synthetic):,}")
print(f"  Columns: {len(df_synthetic.columns)}")
print(f"  Column names: {list(df_synthetic.columns)}")
print(f"\nStrategy distribution:")
print(df_synthetic['reuse_strategy'].value_counts())
print(f"\nDataset ready for training pipeline!")

Exporting synthetic training dataset...
✓ Training dataset successfully saved to: ../data/processed/dataset_optimization_cereal_co2.csv

Final dataset summary:
  Rows: 50,000
  Columns: 8
  Column names: ['subproduct_type', 'season', 'generated_volume_tons', 'moisture_pct', 'process_temperature_c', 'reuse_strategy', 'co2_emissions_kg', 'co2_per_ton']

Strategy distribution:
reuse_strategy
Animal feed           14085
Biochar               13145
Biomass combustion    12208
Composting            10562
Name: count, dtype: int64

Dataset ready for training pipeline!
